# Chapter 9: Working with Network Data
## Volume 1: Foundations | AI for Networking and Security Engineers

This notebook covers the complete journey from raw network data to a queryable knowledge base:

1. **Three-tier data classification** — structured, semi-structured, unstructured
2. **TextFSM/NTC Templates** — parsing CLI output with pre-built templates
3. **LLM extraction** — Haiku for unstructured syslog and edge cases
4. **Multi-vendor normalization** — common schema across Cisco, Juniper, Arista
5. **Vector embeddings** — what they are, why they work, how to build them
6. **Network Knowledge Base** — semantic search over configs, syslogs, runbooks
7. **RAG for networking** — ask plain-English questions, get grounded answers

> Set your API key in Colab Secrets as `ANTHROPIC_API_KEY`

In [ ]:
!pip install anthropic chromadb -q

In [ ]:
import os
import json
import re
import math
from typing import List, Optional
from anthropic import Anthropic

# --- API key loading: Colab Secrets first, getpass fallback for local use ---
try:
    from google.colab import userdata
    _key = userdata.get('ANTHROPIC_API_KEY')
    if _key:
        os.environ['ANTHROPIC_API_KEY'] = _key
except Exception:
    pass

if not os.environ.get('ANTHROPIC_API_KEY'):
    import getpass
    os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('Enter ANTHROPIC_API_KEY: ')

client = Anthropic()
print('Anthropic client ready.')

# Sample network data for demos
CISCO_IOS_CONFIG = """
hostname CORE-RTR-01
!
interface GigabitEthernet0/0
 description TO_DATACENTER
 ip address 10.0.0.1 255.255.255.252
 ip ospf 1 area 0
 no shutdown
!
interface GigabitEthernet0/1
 description TO_BRANCH
 ip address 192.168.1.1 255.255.255.0
 no shutdown
!
router ospf 1
 router-id 1.1.1.1
 area 0 authentication message-digest
 passive-interface default
 no passive-interface GigabitEthernet0/0
!
router bgp 65001
 neighbor 10.0.0.2 remote-as 65002
 neighbor 10.0.0.2 password BGPAUTH123
 neighbor 10.0.0.2 soft-reconfiguration inbound
!
snmp-server community PUBLICREAD ro
snmp-server community PRIVATEWRITE rw
!
line vty 0 4
 transport input ssh
 exec-timeout 10 0
 login local
"""

JUNIPER_INTERFACE_OUTPUT = """
Interface               Admin Link Proto    Local                 Remote
ge-0/0/0                up    up
ge-0/0/0.0              up    up   inet     10.0.0.1/30
ge-0/0/1                up    down
lo0                     up    up
lo0.0                   up    up   inet     192.168.100.1
"""

SYSLOG_SAMPLES = [
    "Jan 22 14:23:01 CORE-RTR-01 %OSPF-5-ADJCHG: Process 1, Nbr 10.0.0.2 on GigabitEthernet0/0 from FULL to DOWN, Neighbor Down: Dead timer expired",
    "Jan 22 14:23:15 CORE-RTR-01 %BGP-5-ADJCHANGE: neighbor 10.0.0.2 Down BGP Notification sent error 2 (Open Message Error), suberror 2 (Bad Peer AS)",
    "Jan 22 14:25:00 CORE-RTR-01 %LINEPROTO-5-UPDOWN: Line protocol on Interface GigabitEthernet0/1, changed state to down",
    "Jan 22 14:26:12 CORE-RTR-01 %SYS-5-CONFIG_I: Configured from console by admin on vty0 (10.100.1.50)",
    "Jan 22 14:30:00 CORE-RTR-01 %OSPF-5-ADJCHG: Process 1, Nbr 10.0.0.2 on GigabitEthernet0/0 from LOADING to FULL, Loading Done",
    "Jan 22 14:35:22 CORE-RTR-01 %SEC-6-IPACCESSLOGP: list MGMT-ACL denied tcp 172.16.100.50(34521) -> 10.0.0.1(22), 1 packet",
]

BGP_SUMMARY_CISCO = """
BGP router identifier 1.1.1.1, local AS number 65001
BGP table version is 45, main routing table version 45

Neighbor        V    AS MsgRcvd MsgSent   TblVer  InQ OutQ Up/Down  State/PfxRcd
10.0.0.2        4 65002    1247    1243       45    0    0 00:18:32       15
10.0.0.6        4 65003     892     889       45    0    0 00:12:44        8
10.0.0.10       4 65004       0       5        0    0    0 00:00:10 Active
"""

print('Sample data loaded.')

---
## Demo 1: The Three-Tier Data Spectrum

Network data falls into three tiers, each requiring a different parsing approach.
The key insight: use the cheapest tool that works for each tier.
Don't use an LLM where TextFSM works. Don't use TextFSM where it breaks.

In [ ]:
# TIER 1: Structured data — just deserialize, no parsing needed
# Example: Cisco DNA Center REST API response

STRUCTURED_API_RESPONSE = {
    "devices": [
        {"hostname": "CORE-RTR-01", "ipAddress": "10.0.0.1", "reachabilityStatus": "Reachable",
         "platformId": "C9300-48P", "softwareVersion": "17.09.04"},
        {"hostname": "BRANCH-RTR-01", "ipAddress": "192.168.1.1", "reachabilityStatus": "Unreachable",
         "platformId": "ISR4331", "softwareVersion": "17.06.05"}
    ]
}

# TIER 1: Just use it directly — no parsing needed
print("=== TIER 1: Structured (REST API JSON) ===")
for device in STRUCTURED_API_RESPONSE["devices"]:
    status = "✓" if device["reachabilityStatus"] == "Reachable" else "✗"
    print(f"  {status} {device['hostname']:20} {device['ipAddress']:15} {device['platformId']}")
print("  → Direct access. Zero parsing code needed.")
print()

# TIER 2: Semi-structured — consistent columns, TextFSM handles it
CISCO_INT_BRIEF = """
Interface              IP-Address      OK? Method Status                Protocol
GigabitEthernet0/0     10.0.0.1        YES NVRAM  up                    up
GigabitEthernet0/1     192.168.1.1     YES NVRAM  up                    up
GigabitEthernet0/2     unassigned      YES NVRAM  administratively down down
Loopback0              1.1.1.1         YES NVRAM  up                    up
"""

print("=== TIER 2: Semi-structured (CLI table output) ===")
# Manual regex parsing to simulate TextFSM behavior (without requiring ntc_templates install)
pattern = re.compile(
    r'^(\S+)\s+(\d+\.\d+\.\d+\.\d+|unassigned)\s+\S+\s+\S+\s+(up|administratively down|down)\s+(up|down)',
    re.MULTILINE
)
matches = pattern.findall(CISCO_INT_BRIEF)
for iface, ip, status, proto in matches:
    indicator = "✓" if status == "up" and proto == "up" else "✗"
    print(f"  {indicator} {iface:30} {ip:15} {status}/{proto}")
print("  → TextFSM template. Consistent for same IOS version.")
print()

# TIER 3: Unstructured syslog — no consistent column structure
print("=== TIER 3: Unstructured (syslog) ===")
for line in SYSLOG_SAMPLES[:3]:
    print(f"  RAW: {line[:80]}...")
print("  → No regex template can handle all 847 Cisco syslog formats.")
print("  → LLM extraction is the right tool here.")

---
## Demo 2: LLM-Based Extraction for Unstructured Syslog

Tier 3 data: syslog messages. 847+ formats, all vendors, all protocols.
We use Haiku (cheap, fast) for extraction — this is simple classification,
not complex reasoning. At 50,000 lines/day, it costs $0.15/month.

In [ ]:
def extract_syslog_event(syslog_line: str) -> dict:
    """Extract structured data from any syslog message using Haiku."""
    prompt = f"""Extract structured data from this network syslog message.

Syslog: {syslog_line}

Return JSON only:
{{"severity": "critical|high|medium|low|info",
 "protocol": "OSPF|BGP|STP|ISIS|interface|security|system|other",
 "event_type": "neighbor_down|neighbor_up|link_down|link_up|config_change|security|other",
 "device_component": "interface or process if visible, else null",
 "peer_address": "IP address if present, else null",
 "summary": "one sentence in plain English"}}"""

    resp = client.messages.create(
        model='claude-haiku-4-5',
        max_tokens=250,
        temperature=0,
        messages=[{'role': 'user', 'content': prompt}]
    )
    raw = resp.content[0].text
    s, e = raw.find('{'), raw.rfind('}') + 1
    return json.loads(raw[s:e]) if s >= 0 else {"error": raw}


print('=== Syslog LLM Extraction Demo ===')
print()

total_cost = 0
for line in SYSLOG_SAMPLES:
    result = extract_syslog_event(line)
    # Estimate cost (Haiku: $1/M input, $5/M output)
    est_cost = (len(line.split()) * 1.3 + 100) / 1e6 * 1.0 + 200 / 1e6 * 5.0
    total_cost += est_cost

    severity_icon = {'critical': '🔴', 'high': '🟠', 'medium': '🟡', 'low': '🟢', 'info': '⚪'}
    icon = severity_icon.get(result.get('severity', 'info'), '⚪')

    print(f"{icon} [{result.get('protocol','?'):10}] [{result.get('event_type','?'):15}] {result.get('summary','?')[:70]}")
    if result.get('peer_address'):
        print(f"   Peer: {result['peer_address']}  Component: {result.get('device_component','?')}")
    print()

print(f'Cost for {len(SYSLOG_SAMPLES)} syslog lines: ~${total_cost:.5f}')
print(f'At 50,000 lines/day: ~${total_cost / len(SYSLOG_SAMPLES) * 50000 * 30:.2f}/month')

---
## Demo 3: Multi-Vendor BGP Extraction

BGP summary output looks different on every vendor.
TextFSM needs one template per vendor/version combination.
LLM extraction handles any vendor in one call.

We use Sonnet here (not Haiku) because understanding BGP state
semantics (Active vs Idle vs Connect) requires vendor context.

In [ ]:
# Simulate different vendor BGP outputs
JUNIPER_BGP_SUMMARY = """
Groups: 2 Peers: 3 Down peers: 1
Table          Tot Paths  Act Paths Suppressed    History Damp State    Pending
inet.0                45         40          0          0          0          0

Peer                     AS      InPkt     OutPkt    OutQ   Flaps Last Up/Dwn State|#Active/Received/Accepted/Damped...
10.0.0.2              65002       1247       1243       0       1       18:32 Established
10.0.0.6              65003        892        889       0       0       12:44 Established
10.0.0.10             65004          0          5       0       2       0:10 Active
"""

# Supported vendors for normalization — anything outside this set raises ValueError
SUPPORTED_VENDORS = {'cisco_ios', 'cisco_nxos', 'juniper_junos', 'arista_eos'}

def normalize(vendor: str) -> str:
    """
    Normalize a vendor string to a canonical key.
    Raises ValueError for unknown vendors so callers get a clear error
    rather than silently producing bad data.
    """
    canonical = vendor.lower().replace(' ', '_').replace('-', '_')
    if canonical not in SUPPORTED_VENDORS:
        raise ValueError(
            f"Unknown vendor '{vendor}'. Supported vendors: {sorted(SUPPORTED_VENDORS)}"
        )
    return canonical


def extract_bgp_neighbors(show_output: str, vendor_hint: str = "unknown") -> List[dict]:
    """Extract BGP neighbors from ANY vendor's show bgp summary."""
    prompt = f"""Extract BGP neighbor data from this {vendor_hint} show command output.

Output:
{show_output}

Return JSON array only:
[{{"neighbor_ip": "x.x.x.x", "remote_as": 12345,
  "state": "Established|Active|Idle|Connect|OpenSent",
  "uptime": "string or null",
  "prefixes_received": integer_or_null,
  "is_up": true_or_false}}]"""

    resp = client.messages.create(
        model='claude-sonnet-4-20250514',
        max_tokens=600,
        temperature=0,
        messages=[{'role': 'user', 'content': prompt}]
    )
    raw = resp.content[0].text
    s, e = raw.find('['), raw.rfind(']') + 1
    return json.loads(raw[s:e]) if s >= 0 else []


print('=== Multi-Vendor BGP Extraction ===')
print()

for vendor, output in [('Cisco IOS', BGP_SUMMARY_CISCO), ('Juniper JunOS', JUNIPER_BGP_SUMMARY)]:
    print(f'--- {vendor} ---')
    neighbors = extract_bgp_neighbors(output, vendor_hint=vendor)
    for n in neighbors:
        status = '✓ UP' if n.get('is_up') else '✗ DOWN'
        pfx = n.get('prefixes_received', '?')
        print(f"  {status}  {n.get('neighbor_ip'):15} AS{n.get('remote_as'):6}  state={n.get('state'):15} up={n.get('uptime','?')} prefixes={pfx}")
    print()

print('Same extraction function, two completely different output formats.')
print('TextFSM would need two separate templates (and would break on version changes).')
print()

# Demonstrate normalize() behaviour
print('=== normalize() vendor validation ===')
for v in ('cisco_ios', 'arista_eos', 'palo_alto'):
    try:
        canonical = normalize(v)
        print(f"  normalize('{v}') -> '{canonical}'")
    except ValueError as exc:
        print(f"  normalize('{v}') -> ValueError: {exc}")

---
## Demo 4: Config Normalization — Common Schema Across Vendors

Before embedding anything, normalize to a common schema.
Every vendor, every data type → same NetworkEvent structure.
This is the translation layer — like IS-IS redistributing into OSPF.

In [ ]:
from dataclasses import dataclass, asdict
from datetime import datetime

@dataclass
class NetworkEvent:
    """Universal normalized network event — vendor-agnostic."""
    event_id: str
    timestamp: str
    source_device: str
    vendor: str
    data_type: str
    severity: str
    raw_data: str
    parsed_fields: dict
    tags: List[str]
    embedding_text: str  # Optimized for semantic search


def syslog_to_event(line: str, device: str, vendor: str) -> Optional[NetworkEvent]:
    """Convert a syslog line to a normalized NetworkEvent."""
    extracted = extract_syslog_event(line)
    embedding_text = (
        f"Network event on {device} ({vendor}): {extracted.get('summary', '')}. "
        f"Protocol: {extracted.get('protocol', '')}. "
        f"Event: {extracted.get('event_type', '')}. "
        f"Peer: {extracted.get('peer_address', '')}"
    )
    return NetworkEvent(
        event_id=f"{device}_{hash(line) % 1_000_000}",
        timestamp=datetime.now().isoformat(),
        source_device=device,
        vendor=vendor,
        data_type='syslog',
        severity=extracted.get('severity', 'info'),
        raw_data=line,
        parsed_fields=extracted,
        tags=[extracted.get('protocol', ''), extracted.get('event_type', ''), device],
        embedding_text=embedding_text
    )


def config_section_to_event(section: str, device: str, vendor: str, section_name: str) -> Optional[NetworkEvent]:
    """Convert a config section to a normalized NetworkEvent."""
    prompt = f"""Config section '{section_name}' on {device} ({vendor}):
{section}

Return JSON: {{"protocol": "...", "purpose": "one sentence", "security_relevant": true/false, "tags": []}}"""

    resp = client.messages.create(
        model='claude-haiku-4-5', max_tokens=200, temperature=0,
        messages=[{'role': 'user', 'content': prompt}]
    )
    raw = resp.content[0].text
    s, e = raw.find('{'), raw.rfind('}') + 1
    meta = json.loads(raw[s:e]) if s >= 0 else {}

    embedding_text = (
        f"Configuration on {device} ({vendor}): {section_name}. "
        f"{meta.get('purpose', '')}. "
        f"Protocol: {meta.get('protocol', '')}. "
        f"Tags: {', '.join(meta.get('tags', []))}"
    )
    return NetworkEvent(
        event_id=f"config_{device}_{hash(section) % 1_000_000}",
        timestamp=datetime.now().isoformat(),
        source_device=device,
        vendor=vendor,
        data_type='config',
        severity='medium' if meta.get('security_relevant') else 'info',
        raw_data=section,
        parsed_fields=meta,
        tags=meta.get('tags', []) + [device, meta.get('protocol', '')],
        embedding_text=embedding_text
    )


# Process all sample data
print('=== Normalization Pipeline Demo ===')
print()

events = []

# Process syslog lines
print('Processing syslog events...')
for line in SYSLOG_SAMPLES[:3]:
    ev = syslog_to_event(line, 'CORE-RTR-01', 'cisco_ios')
    if ev:
        events.append(ev)
        print(f'  [{ev.data_type:6}] [{ev.severity:8}] {ev.parsed_fields.get("summary","")[:60]}')

# Process config sections
print()
print('Processing config sections...')
config_sections = [
    ('snmp_section', 'snmp-server community PUBLICREAD ro\nsnmp-server community PRIVATEWRITE rw'),
    ('ospf_section', 'router ospf 1\n router-id 1.1.1.1\n area 0 authentication message-digest'),
    ('vty_section', 'line vty 0 4\n transport input ssh\n exec-timeout 10 0'),
]

for section_name, content in config_sections:
    ev = config_section_to_event(content, 'CORE-RTR-01', 'cisco_ios', section_name)
    if ev:
        events.append(ev)
        print(f'  [{ev.data_type:6}] [{ev.severity:8}] {ev.parsed_fields.get("purpose","")[:60]}')

print(f'\nTotal normalized events: {len(events)}')
print(f'All events have embedding_text ready for vectorization.')

---
## Demo 5: Vector Embeddings — What They Are and Why They Work

An embedding converts text into a list of numbers (vector) that represents its meaning.
Similar meanings → similar vectors → close together in geometric space.

'BGP neighbor dropped' and 'BGP session lost' → similar vectors.
'NTP server configured' → very different vector.

This demo shows the concept using character n-gram vectors.
In production, use Voyage AI or OpenAI embeddings for 1536-dim vectors.

In [ ]:
def vectorize(text: str, dims: int = 64) -> List[float]:
    """
    Character bigram frequency vector.
    Production: replace with Voyage AI voyage-3 embeddings (1024-dim, semantic).
    This demo version captures some similarity but is not semantically deep.
    """
    text = text.lower().strip()
    vec = [0.0] * dims
    for i in range(len(text) - 1):
        bigram = text[i:i+2]
        idx = (ord(bigram[0]) * 31 + ord(bigram[1])) % dims
        vec[idx] += 1.0
    total = sum(vec) or 1.0
    return [v / total for v in vec]


def cosine_sim(v1: List[float], v2: List[float]) -> float:
    dot = sum(a * b for a, b in zip(v1, v2))
    m1 = math.sqrt(sum(a**2 for a in v1))
    m2 = math.sqrt(sum(b**2 for b in v2))
    return dot / (m1 * m2) if m1 * m2 > 0 else 0.0


# Demonstrate semantic clustering
test_phrases = [
    # Group A: BGP events
    ('BGP neighbor 10.0.0.2 went down', 'bgp'),
    ('BGP session to peer 10.0.0.2 dropped', 'bgp'),
    ('Routing protocol adjacency lost with 10.0.0.2', 'bgp'),
    # Group B: OSPF events
    ('OSPF neighbor adjacency changed to DOWN', 'ospf'),
    ('OSPF process 1 neighbor dead timer expired', 'ospf'),
    # Group C: Unrelated
    ('NTP server synchronization configured', 'ntp'),
    ('VTY line access restricted to SSH only', 'security'),
]

print('=== Vector Similarity Demo ===')
print('(Higher similarity = more semantically related)')
print()

query = 'BGP session failure with peer 10.0.0.2'
query_vec = vectorize(query)
print(f'Query: "{query}"')
print()

results = []
for phrase, group in test_phrases:
    sim = cosine_sim(query_vec, vectorize(phrase))
    results.append((sim, phrase, group))

results.sort(key=lambda x: -x[0])
for sim, phrase, group in results:
    bar = '█' * int(sim * 20)
    print(f'  {sim:.3f} {bar:20} [{group:8}] {phrase}')

print()
print('Note: In production with 1024-dim Voyage AI embeddings,')
print('"Routing protocol adjacency lost" would score as high as direct BGP matches.')
print('Character n-grams capture surface similarity; neural embeddings capture semantics.')

---
## Demo 6: Network Knowledge Base with ChromaDB

Now we build the actual knowledge base.
ChromaDB stores our vectors + documents.
We can query it semantically — find relevant network data by meaning.

This is the foundation of RAG for networking.

In [ ]:
import chromadb

# Initialize ChromaDB (in-memory for Colab)
chroma = chromadb.Client()
collection = chroma.get_or_create_collection(
    name='network_kb',
    metadata={'hnsw:space': 'cosine'}  # Cosine similarity for text
)

# Prepare network data chunks for the knowledge base
# Each chunk has: embedding_text (for vectorization), content (raw data), metadata

kb_chunks = []

# Add normalized events we created earlier
for ev in events:
    kb_chunks.append({
        'id': ev.event_id,
        'embedding_text': ev.embedding_text,
        'content': ev.raw_data[:500],
        'metadata': {
            'device': ev.source_device,
            'vendor': ev.vendor,
            'data_type': ev.data_type,
            'severity': ev.severity
        }
    })

# Add config sections directly
config_sections_full = [
    ('bgp_section', 'router bgp 65001\n neighbor 10.0.0.2 remote-as 65002\n neighbor 10.0.0.2 password BGPAUTH123'),
    ('acl_missing_deny', 'ip access-list extended PERMIT-ALL\n permit ip any any'),
    ('ospf_auth', 'router ospf 1\n area 0 authentication message-digest\n router-id 1.1.1.1'),
    ('snmp_v2', 'snmp-server community PUBLICREAD ro\nsnmp-server community PRIVATEWRITE rw\nsnmp-server host 10.100.1.10 PUBLICREAD'),
]

for section_id, content in config_sections_full:
    protocol = 'BGP' if 'bgp' in section_id else ('OSPF' if 'ospf' in section_id else ('SNMP' if 'snmp' in section_id else 'ACL'))
    kb_chunks.append({
        'id': f'cfg_{section_id}',
        'embedding_text': f'Configuration CORE-RTR-01 cisco_ios: {section_id} section. Protocol {protocol}. Content: {content[:200]}',
        'content': content,
        'metadata': {'device': 'CORE-RTR-01', 'vendor': 'cisco_ios', 'data_type': 'config', 'section': section_id}
    })

# Generate embeddings and store in ChromaDB
ids = [c['id'] for c in kb_chunks]
embeddings = [vectorize(c['embedding_text']) for c in kb_chunks]
documents = [c['content'] for c in kb_chunks]
metadatas = [c['metadata'] for c in kb_chunks]

collection.upsert(ids=ids, embeddings=embeddings, documents=documents, metadatas=metadatas)

print(f'Network Knowledge Base built: {collection.count()} records')
print()
print('Record breakdown:')
data_types = {}
for chunk in kb_chunks:
    dt = chunk['metadata'].get('data_type', 'unknown')
    data_types[dt] = data_types.get(dt, 0) + 1
for dt, count in data_types.items():
    print(f'  {dt:10}: {count} records')

In [ ]:
# Semantic search over the knowledge base

def semantic_search(query: str, n: int = 3) -> List[dict]:
    """Semantic search over the network knowledge base."""
    q_vec = vectorize(query)
    results = collection.query(
        query_embeddings=[q_vec],
        n_results=n,
        include=['documents', 'metadatas', 'distances']
    )
    return [
        {
            'content': results['documents'][0][i],
            'metadata': results['metadatas'][0][i],
            'similarity': 1 - results['distances'][0][i]
        }
        for i in range(len(results['documents'][0]))
    ]


test_queries = [
    'BGP neighbor problems and failures',
    'SNMP community string security issues',
    'OSPF authentication configuration',
    'ACL missing deny statement',
]

print('=== Semantic Search Queries ===')
for query in test_queries:
    print(f'Query: "{query}"')
    results = semantic_search(query, n=2)
    for r in results:
        meta = r['metadata']
        print(f'  [{r["similarity"]:.3f}] {meta["device"]} | {meta["data_type"]:6} | {r["content"][:70]}...')
    print()

---
## Demo 7: RAG — Ask Your Network Questions in Plain English

This is the full RAG loop:
1. Embed the question
2. Retrieve relevant network data from ChromaDB
3. Build a grounded prompt: context + question
4. Claude answers using ONLY your actual network data
5. Return answer + citations

No hallucination. Every answer is grounded in what you indexed.
Every answer cites the source so you can verify.

In [ ]:
def ask_network(question: str, n_context: int = 4) -> dict:
    """
    RAG: answer a question about the network using retrieved context.

    Steps:
    1. Semantic search for relevant network data
    2. Build grounded prompt
    3. Generate answer (Sonnet — needs reasoning, not just extraction)
    4. Return answer + source citations
    """
    # Step 1: Retrieve relevant context
    retrieved = semantic_search(question, n=n_context)

    if not retrieved:
        return {'answer': 'No relevant data in knowledge base.', 'sources': []}

    # Step 2: Build context string
    # Put highest-similarity chunks first AND last (mitigate lost-in-the-middle)
    context_parts = []
    for i, r in enumerate(retrieved):
        meta = r['metadata']
        context_parts.append(
            f"[Source {i+1}: device={meta.get('device')} | type={meta.get('data_type')} | section={meta.get('section', 'N/A')} | relevance={r['similarity']:.2f}]\n"
            f"{r['content'][:600]}"
        )

    context_text = '\n\n'.join(context_parts)

    # Step 3: Grounded RAG prompt
    # Key instruction: ONLY use provided context, always cite sources
    rag_prompt = f"""You are a senior network engineer answering questions about a specific network.

RULES:
- Answer ONLY from the context below — no assumptions, no general knowledge
- If the answer is not in the context: say "Not available in current network data"
- Always cite which [Source N] you used
- Be specific: use exact IP addresses, interface names, and config values from the context

NETWORK DATA CONTEXT:
{context_text}

QUESTION: {question}

Answer (with [Source N] citations):"""

    # Step 4: Generate
    resp = client.messages.create(
        model='claude-sonnet-4-20250514',
        max_tokens=600,
        temperature=0,
        messages=[{'role': 'user', 'content': rag_prompt}]
    )

    return {
        'answer': resp.content[0].text,
        'sources': [
            {'rank': i+1, 'device': r['metadata'].get('device'),
             'type': r['metadata'].get('data_type'), 'similarity': r['similarity']}
            for i, r in enumerate(retrieved)
        ],
        'retrieved': len(retrieved)
    }


# Ask questions about the network
questions = [
    'What BGP events happened on CORE-RTR-01?',
    'Is SNMP configured securely on this device?',
    'What OSPF authentication is configured?',
    'Are there any ACLs missing the deny-all at the end?',
]

print('=== RAG: Natural Language Network Queries ===')
print()

for question in questions:
    print(f'Q: {question}')
    result = ask_network(question)
    print(f'A: {result["answer"][:300]}...' if len(result['answer']) > 300 else f'A: {result["answer"]}')
    print(f'Sources: {[(s["rank"], s["type"], s["similarity"][:3] if isinstance(s["similarity"],str) else round(s["similarity"],2)) for s in result["sources"]]}')
    print()

---
## Summary: The Complete Network Data Journey

```
Raw Network Data
    ├── Structured (REST API JSON)      → Direct use, no parsing
    ├── Semi-structured (CLI tables)    → TextFSM / NTC Templates
    └── Unstructured (syslog, edge cases) → LLM extraction (Haiku, $0.15/month)
                  ↓
    Normalize to NetworkEvent (common schema, embedding_text)
                  ↓
    Vectorize embedding_text → store in ChromaDB
                  ↓
    Semantic Search → RAG → Grounded Answers with Citations
```

### Key Rules

- **Don't use LLMs for Tier 2** — TextFSM is free, fast, deterministic
- **Don't use regex for Tier 3** — it breaks with every firmware update
- **Normalize before embedding** — the `embedding_text` field is the secret sauce
- **Vectors are GPS coordinates for meaning** — similar meanings cluster together
- **RAG prevents hallucination** — Claude answers from your data, not from training
- **Always include source citations** — verifiable answers build trust

### What's Next?

Volume 2: **Advanced RAG Architecture**
- Query transformation and HyDE (Hypothetical Document Embeddings)
- Reranking with cross-encoders
- Agentic RAG with multi-step reasoning
- GraphRAG for multi-hop network topology queries
- Production deployment patterns

*Chapter 9 | Volume 1: Foundations | AI for Networking and Security Engineers*